In [ ]:
import numpy as np
from scipy.optimize import brentq


# ---------------------------------------
# basic functions
# ---------------------------------------

def effective_drift(mu, sigma):
    return mu - 0.5 * sigma**2


def conditional_start_log(x0, a, b):

    loga = np.log(a)
    logb = np.log(b)
    y0 = np.log(x0)

    if y0 < loga:
        return loga

    if y0 > logb:
        return logb

    return y0


def hit_probability_interval(x0, mu, a, b, sigma):

    nu = effective_drift(mu, sigma)

    if abs(nu) < 1e-14:
        return 1.0

    y0 = np.log(x0)
    loga = np.log(a)
    logb = np.log(b)

    if nu > 0 and y0 > logb:
        return np.exp(-2 * nu * (y0 - logb) / sigma**2)

    if nu < 0 and y0 < loga:
        return np.exp(2 * nu * (loga - y0) / sigma**2)

    return 1.0


def zero_mass_probability(x0, mu, a, b, sigma):

    p_hit = hit_probability_interval(x0, mu, a, b, sigma)

    return 1 - p_hit

In [ ]:
def simulate_conditional_occupation(
    x0,
    mu,
    a,
    b,
    sigma,
    dt,
    t_max,
    n_paths,
    stop_return_prob=1e-8,
    seed=123,
):

    np.random.seed(seed)

    loga = np.log(a)
    logb = np.log(b)

    nu = effective_drift(mu, sigma)

    y_start = conditional_start_log(x0, a, b)

    n_steps = int(t_max / dt)

    samples = np.zeros(n_paths)

    if abs(nu) > 1e-14:
        stop_margin = sigma**2 / (2 * abs(nu)) * np.log(1 / stop_return_prob)
    else:
        stop_margin = np.inf

    for i in range(n_paths):

        y = y_start
        occ = 0.0

        for step in range(n_steps):

            z = np.random.normal()

            y_new = y + nu * dt + sigma * np.sqrt(dt) * z

            y_mid = 0.5 * (y + y_new)

            if loga <= y_mid <= logb:
                occ = occ + dt

            y = y_new

            if nu > 0 and y > logb + stop_margin:
                break

            if nu < 0 and y < loga - stop_margin:
                break

        samples[i] = occ

    return samples

In [ ]:
def conditional_laplace_transform(p, x0, mu, a, b, sigma):

    loga = np.log(a)
    logb = np.log(b)

    L = logb - loga

    nu = effective_drift(mu, sigma)

    if abs(nu) < 1e-14:
        raise ValueError("Critical case is treated separately.")

    y_start = conditional_start_log(x0, a, b)
    z0 = y_start - loga

    root = np.sqrt(nu**2 + 2 * sigma**2 * p + 0j)

    r1 = (-nu + root) / sigma**2
    r2 = (-nu - root) / sigma**2

    if nu > 0:

        k = 2 * nu / sigma**2

        M = np.array([
            [r1, r2],
            [(r1 + k) * np.exp(r1 * L), (r2 + k) * np.exp(r2 * L)]
        ], dtype=complex)

        rhs = np.array([0, k], dtype=complex)

    else:

        k = -2 * nu / sigma**2

        M = np.array([
            [r1 - k, r2 - k],
            [r1 * np.exp(r1 * L), r2 * np.exp(r2 * L)]
        ], dtype=complex)

        rhs = np.array([-k, 0], dtype=complex)

    c1, c2 = np.linalg.solve(M, rhs)

    value = c1 * np.exp(r1 * z0) + c2 * np.exp(r2 * z0)

    return value

In [ ]:
def theta_equation(theta, L, rho):

    return (rho**2 - theta**2) * np.sin(theta * L) + 2 * rho * theta * np.cos(theta * L)


def find_theta_roots(a, b, mu, sigma, n_roots):

    L = np.log(b / a)

    nu = effective_drift(mu, sigma)

    if abs(nu) < 1e-14:
        raise ValueError("There are no transient roots in the critical case.")

    rho = abs(nu) / sigma**2

    theta_max = (n_roots + 6) * np.pi / L

    grid = np.linspace(1e-10, theta_max, 50000)

    values = theta_equation(grid, L, rho)

    roots = []

    for i in range(len(grid) - 1):

        if len(roots) >= n_roots:
            break

        left = grid[i]
        right = grid[i + 1]

        f_left = values[i]
        f_right = values[i + 1]

        if f_left * f_right < 0:

            root = brentq(
                lambda th: theta_equation(th, L, rho),
                left,
                right
            )

            if root > 1e-8:
                roots.append(root)

    return np.array(roots)

In [ ]:
def residue_coefficients(x0, mu, a, b, sigma, n_terms=50):

    theta = find_theta_roots(a, b, mu, sigma, n_terms)

    nu = effective_drift(mu, sigma)

    lambdas = 0.5 * sigma**2 * theta**2 + 0.5 * nu**2 / sigma**2

    coeffs = np.zeros(len(lambdas))

    for i in range(len(lambdas)):

        lam = lambdas[i]

        eps = 1e-7 * max(1, abs(lam))

        p = -lam + eps

        value = conditional_laplace_transform(p, x0, mu, a, b, sigma)

        coeffs[i] = np.real(eps * value)

    return lambdas, coeffs

In [ ]:
def conditional_density(u_values, x0, mu, a, b, sigma, n_terms=50):

    u_values = np.array(u_values)

    lambdas, coeffs = residue_coefficients(
        x0,
        mu,
        a,
        b,
        sigma,
        n_terms
    )

    density = np.zeros(len(u_values))

    for i in range(len(u_values)):

        u = u_values[i]

        total = 0.0

        for n in range(len(lambdas)):

            total = total + coeffs[n] * np.exp(-lambdas[n] * u)

        density[i] = total

    density[np.abs(density) < 1e-12] = 0.0

    return density

In [ ]:
def critical_scaling_density(z_values, a, b, sigma):

    z_values = np.array(z_values)

    L = np.log(b / a)

    density = np.sqrt(2) * sigma / (L * np.sqrt(np.pi))

    density = density * np.exp(-sigma**2 * z_values**2 / (2 * L**2))

    density[z_values < 0] = 0.0

    return density

In [ ]:
def histogram_points(samples, bins):

    samples = np.array(samples)

    samples = samples[np.isfinite(samples)]

    samples = samples[samples > 0]

    hist, edges = np.histogram(samples, bins=bins, density=True)

    centers = 0.5 * (edges[:-1] + edges[1:])

    return centers, hist

In [ ]:
a = 1.0
b = 2.0
sigma = 1.0

x0 = 0.7
mu = 0.0

dt = 0.01
t_max = 1000
n_paths = 10000

samples = simulate_conditional_occupation(
    x0,
    mu,
    a,
    b,
    sigma,
    dt,
    t_max,
    n_paths
)

u_grid = np.linspace(0.02, 8, 500)

theory = conditional_density(
    u_grid,
    x0,
    mu,
    a,
    b,
    sigma,
    n_terms=50
)

bins = np.linspace(0.02, 8, 60)

centers, hist = histogram_points(samples, bins)

p0 = zero_mass_probability(x0, mu, a, b, sigma)

print("zero mass =", p0)
print("mean simulated occupation =", samples.mean())